# LangChain 기초 + MLflow 트레이싱
Claude 모델 호출 → 프롬프트 템플릿 → 구조화 출력 → 도구 호출까지.  
모든 호출은 `mlflow.langchain.autolog()` 로 https://mlflow.dove-nest.com 에 트레이스로 남는다.

# **1. 환경 설정**

In [ ]:
from common import setup_mlflow, get_model

setup_mlflow()          # MLflow 연결 + autolog
model = get_model()     # 기본 claude-opus-5 (.env 의 CLAUDE_MODEL 로 변경)
model.model

# **2. 모델 호출**
## **2-1. invoke**
채팅 모델은 메시지 리스트를 받고 `AIMessage` 를 돌려준다. 문자열 하나만 넘기면 HumanMessage 로 감싸진다.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

res = model.invoke([
    SystemMessage("너는 간결하게 답하는 한국어 조교야."),
    HumanMessage("LangChain 과 LangGraph 의 차이를 두 문장으로 설명해줘."),
])
print(res.text)
res.usage_metadata

## **2-2. stream**

In [ ]:
for chunk in model.stream("RAG 가 뭔지 세 줄로 설명해줘."):
    print(chunk.text, end="", flush=True)

# **3. 프롬프트 템플릿과 체인 (LCEL)**
`|` 로 Runnable 을 이어 붙인다. MLflow 트레이스에서 각 단계가 span 으로 보인다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {domain} 전문가야. 초보자에게 설명하듯 답해."),
    ("human", "{question}"),
])
chain = prompt | model | StrOutputParser()

print(chain.invoke({"domain": "데이터베이스", "question": "인덱스는 왜 쓰나요?"}))

# **4. 구조화 출력**
Pydantic 스키마를 주면 파싱된 객체로 받는다.

In [ ]:
from pydantic import BaseModel, Field

class Paper(BaseModel):
    title: str = Field(description="논문 제목")
    year: int = Field(description="발표 연도")
    key_idea: str = Field(description="핵심 아이디어 한 문장")

structured = model.with_structured_output(Paper)
paper = structured.invoke("Attention Is All You Need 논문 정보를 알려줘.")
paper

# **5. 도구 호출 (Tool calling)**
`@tool` 로 함수를 도구로 만들고 `bind_tools` 로 모델에 연결한다.  
모델은 도구를 **직접 실행하지 않고** 호출 요청(`tool_calls`)만 돌려준다. 실행 루프는 02 노트북에서 LangGraph 로 만든다.

In [ ]:
from langchain.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱한다."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 알려준다."""
    return f"{city}: 맑음, 21도"  # 실습용 가짜 데이터

model_with_tools = model.bind_tools([multiply, get_weather])
ai = model_with_tools.invoke("서울 날씨 알려주고, 37.5 곱하기 4 도 계산해줘.")
ai.tool_calls

In [ ]:
# 도구를 직접 실행하고 결과를 돌려주면 최종 답변을 받는다
from langchain.messages import HumanMessage

tools = {"multiply": multiply, "get_weather": get_weather}
messages = [HumanMessage("서울 날씨 알려주고, 37.5 곱하기 4 도 계산해줘."), ai]
for call in ai.tool_calls:
    messages.append(tools[call["name"]].invoke(call))  # ToolMessage 반환

print(model_with_tools.invoke(messages).text)

# **6. MLflow 에서 트레이스 확인**
UI: https://mlflow.dove-nest.com → Experiments → `langchain-study` → **Traces** 탭  
코드로도 조회할 수 있다.

In [ ]:
import mlflow

traces = mlflow.search_traces(max_results=10)
traces[["trace_id", "request_time", "state", "execution_duration"]]

# **99. 참고자료**
- LangChain 문서: https://docs.langchain.com/oss/python/langchain/overview
- ChatAnthropic: https://docs.langchain.com/oss/python/integrations/chat/anthropic
- MLflow LangChain 트레이싱: https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain